[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/weaviate/recipes/blob/main/query-agent/search-mode/search-mode-get-started.ipynb)

# Query Agent Search Mode - Minimal Example

The Weaviate [Query Agent](https://weaviate.io/developers/agents/query) also supports a **search-only mode**. Instead of generating a final natural language answer, `agent.search()` interprets your natural language query, plans and runs the underlying searches, and returns the ranked result objects directly - ideal when you want to feed the results into your own application logic or UI.

This example uses the products dataset from [`datasets/1k_products.csv`](https://github.com/weaviate/recipes/blob/main/datasets/1k_products.csv) (properties: `name`, `description`, `url`), and assumes it's already loaded into a Weaviate collection named `Products`.

In [ ]:
!pip install weaviate-client[agents]

In [ ]:
import os
from getpass import getpass

if "WEAVIATE_API_KEY" not in os.environ:
    os.environ["WEAVIATE_API_KEY"] = getpass("Weaviate API Key")
if "WEAVIATE_URL" not in os.environ:
    os.environ["WEAVIATE_URL"] = getpass("Weaviate URL")

In [ ]:
import weaviate
from weaviate.auth import Auth

client = weaviate.connect_to_weaviate_cloud(
    cluster_url=os.environ.get("WEAVIATE_URL"),
    auth_credentials=Auth.api_key(os.environ.get("WEAVIATE_API_KEY")),
)

## Run a Search

Create a `QueryAgent` with access to the `Products` collection and call `search()` with a natural language query. The response contains the retrieved objects in `search_results`, along with the underlying `searches` the agent decided to run.

In [ ]:
from weaviate.agents.query import QueryAgent

agent = QueryAgent(client=client, collections=["Products"])

response = agent.search("waterproof shoes for hiking", limit=5)

for obj in response.search_results.objects:
    print(obj.properties["name"])
    print(obj.properties["description"])
    print()

You can also inspect how the agent translated your query into searches:

In [ ]:
print(response.searches)

## Paginate Through Results

`response.next()` reuses the same underlying searches, so you can page through a consistent result set without the agent re-planning the query.

In [ ]:
next_page = response.next(limit=5, offset=5)

for obj in next_page.search_results.objects:
    print(obj.properties["name"])

In [ ]:
client.close()